# Prompting and structured output

## Goal

Compare system prompts, few-shot examples, requested text formatting and schema-based output.

## Setup

Select the **Python (LangChain OpenAI)** kernel and run cells from top to bottom.
`OPENAI_API_KEY` and `OPENAI_MODEL` are read only from this course's root `.env`.
Shared settings are in `lesson_support.py`: up to 1024 output tokens, a 30-second
HTTP timeout, no HTTP retries, and LangSmith tracing disabled.
`make_model("another-api-model-id")` creates a different model with the same key.
The default text settings are intended for `gpt-5.6-luna`.
Creating a model does not call the API; `invoke` and consuming `stream` make paid requests.


In [ ]:
from pathlib import Path
import sys

# Find this course's root, whether Jupyter starts at the root or in notebooks/.
current_dir = Path.cwd().resolve()
course_root = next(
    (folder for folder in (current_dir, *current_dir.parents)
     if (folder / "course.json").is_file() and (folder / "lesson_support.py").is_file()),
    None,
)
if course_root is None:
    raise RuntimeError("Start Jupyter from the course root, as described in README.md.")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from lesson_support import make_model

model = make_model()
print("Model:", model.model_name)


In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=model)

question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

In [ ]:
system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Few-shot examples

In [ ]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Structured prompts

In [ ]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Structured output

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

Passing `CapitalInfo` lets LangChain select a structured-output strategy.
To explicitly request provider-native schema enforcement, use
`response_format=ProviderStrategy(CapitalInfo, strict=True)` after importing
`ProviderStrategy` from `langchain.agents.structured_output`.
This controls schema conformance, not factual correctness; refusals and incomplete replies still need handling.

In [ ]:
response["structured_response"].name

In [ ]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")